In [1]:
from system_check import *
CUDA_state_print_limited()

CUDA доступна: True
Количество GPU: 4


In [2]:
import torch

def get_device() -> str:
    """
    Возвращает 'cuda' если GPU доступен, иначе 'cpu'.
    """
    return "cuda:1" if torch.cuda.is_available() else "cpu"

In [3]:
device = get_device()

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc

# Очистка памяти от предыдущих моделей
gc.collect()
torch.cuda.empty_cache()

# Загрузка ТОЛЬКО на CPU (без accelerate/device_map)
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token  # критически важно для Qwen

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,      # экономия памяти
    low_cpu_mem_usage=True,         # загрузка без переполнения ОЗУ
    trust_remote_code=False         # не требуется для Qwen2.5
).to(device)  # ← ЯВНОЕ перемещение на CPU

print("✅ Модель загружена. Память:", sum(p.numel() for p in model.parameters()) / 1e9, "B параметров")

/home/pmartynyuk/UnTIE project/untie_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.40it/s]


In [ ]:
from transformers import *
roberta_qa_dir = "/home/pmartynyuk/UnTIE project/scripts/models_processing/models/rubert_ru_qa_model"

model = AutoModelForQuestionAnswering.from_pretrained(roberta_qa_dir, output_attentions=True).to("cuda:1")
concrete_tokenizer = AutoTokenizer.from_pretrained(roberta_qa_dir)

In [ ]:
# import sys
# sys.path.append("/global_functions")
# from global_functions.global_functions import *

In [ ]:
from models_processing.models_setting import *
get_concrete_tokenizer()

In [ ]:
from transformers import *

device =  "cuda:2"
concrete_qa_model = AutoModelForQuestionAnswering.from_pretrained("/home/pmartynyuk/UnTIE project/scripts/models_processing/models/rubert_ru_qa_model").to(device)


In [ ]:
from models_processing.models_setting import *
set_concrete_QA_model(SupportedLanguage.RUS, SupportedModels.BERT)

In [ ]:
set_concrete_QA_model(SupportedLanguage.RUS, SupportedModels.BERT)

In [ ]:
set_concrete_QA_model(SupportedLanguage.RUS, SupportedModels.BERT)
get_concrete_QA_model()

In [ ]:
df = pd.read_csv('/home/pmartynyuk/UnTIE project/datasets/ruserrc_task.csv', 
                    encoding='utf-8', 
                    sep=';')
df

In [ ]:
# Результаты
datasets_results_path = "../datasets_results"
res_name = "results_keys_rus_01.json"

# Модель документа
model_params_path = "../model_params"
model_name = "ruserrc_init_model.json"

In [ ]:
# Чтение параметров модели
df_model_params = pd.read_json(f"{model_params_path}/{model_name}")

field_name = df_model_params["fields"][0]["field_name"]
questions = df_model_params["fields"][0]["questions"]
keywords = df_model_params["fields"][0]["keywords"]
text = df_model_params["fields"][0]["text"]

In [ ]:
questions

In [ ]:
# Подключение логгера

from logs_processing.logger_config import setup_logger

logger = setup_logger("main", "05_RU_Dataset_untie.log")
extra_logger = setup_logger("details", "05_extra_RU_Dataset_untie.log")


In [ ]:
num_of_sample = 0

In [ ]:
them_aspect = ThematicAspect(field_name, convert_into_question_class(questions))
filtering_set = FilteringSet(text, keywords)

In [ ]:
text_of_doc = df["text_clean"][num_of_sample]

import re

# Через регулярное выражение
text_of_doc = re.sub(r'\s+', ' ', text_of_doc).strip()
text_of_doc

In [ ]:
reference_answers = df["Task_aspects"][num_of_sample]
reference_answers

In [ ]:
# Делим на чанки
text_proc = RuTextProcesser()
sents = text_proc.split_into_sentences(text_of_doc)